In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "schmelz2011chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Schmelz_2011_Inference Chess Raw Data.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)


In [3]:

df['study_id']="schmelz2011chimpanzees"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

In [4]:
df=df.rename(columns={"group": "group_original", 
    "subject": "ape"})

In [5]:
df['ape'] = df['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
# df.columns

ape_pairs = [['alexandra', 'annett'],
            ['annett','alex'],
            ['alex','fifi' ],
            ['fifi','jahaga'],
            ['jahaga', 'gertrudia'],
            ['gertrudia','alexandra'],
            ['patrick','pia'],
            ['pia','frodo'],
            ['frodo','sandra'],
            ['sandra', 'lome'],
            ['tai','patrick'],
            ['lome','tai']]
for x,y in ape_pairs:
    df.loc[df.ape == x, ['ape_2']] = y


comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

df['dyad']=df.ape.str.cat(df.ape_2, sep='_')
df['role']="focal_participant"
df['role_2']="competitor"

In [6]:
df.rename(columns={"ape": "participant", "ape_2": "participant_2"}, inplace=True)
gap_list = ['condition','choice']
for x in gap_list:
    df[x].replace(' ', '_', inplace=True, regex=True)

In [7]:
schmelz2011chimpanzees_standardized=df[[ 'study_id', 'participant', 'sex', 'role', 
        'participant_2', 'sex_2', 'role_2',  'species', 'dyad','group_original', 
        'session', 'trial', 'condition',  'choice'  ]]

In [8]:
comp_out_path_stand = os.path.join(out_pathway, 'schmelz2011chimpanzees_standardized.csv')
schmelz2011chimpanzees_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


In [9]:
names =schmelz2011chimpanzees_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
schmelz2011chimpanzees_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'schmelz2011chimpanzees_glossary.csv')
schmelz2011chimpanzees_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
